# 🎵 IsaiCraft — Remote GPU Processing Engine (Tier 3)

**An Intelligent Audio Processing Ecosystem for AI-Powered Music Production**

This notebook contains the **original Tier 3 Google Colab GPU Processing Engine** for IsaiCraft. It receives user vocal recordings and music metadata from the frontend via a secure ngrok tunnel, generates AI instrumental backing tracks using Meta's **MusicGen-Small**, cleans and pitch-corrects vocals using **PyWorld (55% humanized soft-pitch correction)**, processes tracks through **Spotify Pedalboard** vocal and instrumental chains, and delivers dynamic mixbus mastered audio as Base64-encoded WAV stems to the React client.

In [ ]:
# @title 1. Install Required Dependencies
!pip install -q transformers scipy librosa pyworld noisereduce pedalboard pyngrok fastapi uvicorn python-multipart nest-asyncio soundfile torch torchaudio accelerate

In [ ]:
# @title 2. Core Python Imports
import io
import os
import base64
import tempfile
import numpy as np
import scipy.signal
import soundfile as sf
import librosa
import pyworld as pw
import noisereduce as nr
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from pedalboard import (
    Pedalboard,
    HighpassFilter,
    PeakFilter,
    HighShelfFilter,
    Compressor,
    Reverb,
    Gain
)
import nest_asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok

In [ ]:
# @title 3. GPU Hardware Detection
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    !nvidia-smi
else:
    print("WARNING: GPU is not active. Switch runtime to GPU (Runtime -> Change runtime type -> T4 GPU).")

In [ ]:
# @title 4. Load & Cache MusicGen-Small Model
print("Loading Meta MusicGen-Small model...")
music_processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
music_model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)
music_sample_rate = music_model.config.audio_encoder.sampling_rate
print(f"MusicGen loaded successfully on {device} (Sampling Rate: {music_sample_rate} Hz)")

In [ ]:
# @title 5. DSP & Audio Processing Functions
def humanized_autotune(y, sr, retune_strength=0.55):
    """
    Humanized Soft-Pitch Correction using PyWorld vocoder.
    Extracts fundamental frequency (F0) via DIO + StoneMask, smooths with median filter,
    and fractionally pulls pitch toward the nearest chromatic 12-TET semitone.
    """
    x = y.astype(np.float64)
    _f0, t = pw.dio(x, sr, frame_period=5.0)
    f0 = pw.stonemask(x, _f0, t, sr)
    sp = pw.cheaptrick(x, f0, t, sr)
    ap = pw.d4c(x, f0, t, sr)

    smoothed_f0 = scipy.signal.medfilt(f0, kernel_size=5)
    tuned_f0 = np.copy(smoothed_f0)

    for i, f in enumerate(smoothed_f0):
        if f > 50.0:  # Voiced threshold
            midi_note = 69 + 12 * np.log2(f / 440.0)
            snapped_midi = np.round(midi_note)
            target_f = 440.0 * (2.0 ** ((snapped_midi - 69) / 12.0))
            tuned_f0[i] = f + ((target_f - f) * retune_strength)

    y_tuned = pw.synthesize(tuned_f0, sp, ap, sr, frame_period=5.0)
    return y_tuned.astype(np.float32)

def normalize_audio(audio_array, target_peak):
    """Normalizes audio array to target peak amplitude."""
    max_val = np.max(np.abs(audio_array))
    if max_val > 0:
        return (audio_array / max_val) * target_peak
    return audio_array

def audio_to_base64(audio_array, sr):
    """Converts a float NumPy audio array to a Base64-encoded WAV data URL."""
    buffer = io.BytesIO()
    clipped = np.clip(audio_array, -1.0, 1.0)
    sf.write(buffer, clipped, sr, format="WAV", subtype="PCM_16")
    buffer.seek(0)
    b64 = base64.b64encode(buffer.read()).decode("utf-8")
    return f"data:audio/wav;base64,{b64}"

In [ ]:
# @title 6. Define FastAPI GPU Microservice
app = FastAPI(title="IsaiCraft GPU Engine", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
async def health_check():
    return {"status": "online", "engine": "IsaiCraft GPU Processing Node", "device": device}

@app.post("/generate")
async def generate(
    project_name: str = Form("Unnamed"),
    mood: str = Form("Dynamic performance"),
    genre: str = Form("Unknown"),
    lyrics: str = Form(""),
    vocal_file: UploadFile = File(...)
):
    try:
        # 1. MusicGen Instrumental Backing Track Generation
        prompt = f"{genre} track with {mood}. Theme: {lyrics[:100]}"
        music_inputs = music_processor(text=[prompt], padding=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            audio_values = music_model.generate(**music_inputs, max_new_tokens=1500)
        
        music_raw = audio_values[0, 0].cpu().numpy().astype(np.float32)

        # 2. Instrumental EQ Shaping (Pedalboard)
        instrumental_board = Pedalboard([
            PeakFilter(cutoff_frequency_hz=2500, gain_db=-6.0, q=1.0),
            HighpassFilter(cutoff_frequency_hz=40)
        ])
        music_shaped = instrumental_board(music_raw, music_sample_rate)

        # 3. Vocal Audio Loading via Librosa
        vocal_bytes = await vocal_file.read()
        if not vocal_bytes:
            raise HTTPException(status_code=400, detail="Empty vocal file received")
        
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as temp_vocal:
            temp_vocal.write(vocal_bytes)
            temp_vocal_path = temp_vocal.name

        try:
            y, sr = librosa.load(temp_vocal_path, sr=music_sample_rate, mono=True)
        finally:
            if os.path.exists(temp_vocal_path):
                os.remove(temp_vocal_path)

        # 4. Noise Reduction
        y_denoised = nr.reduce_noise(y=y, sr=sr, prop_decrease=0.85, stationary=False)

        # 5. Humanized Pitch Correction (retune_strength=0.55)
        y_tuned = humanized_autotune(y_denoised, sr, retune_strength=0.55)

        # 6. Vocal DSP Processing (Pedalboard FX Chain)
        vocal_board = Pedalboard([
            HighpassFilter(cutoff_frequency_hz=100),
            PeakFilter(cutoff_frequency_hz=300, gain_db=-3.0, q=1.0),
            HighShelfFilter(cutoff_frequency_hz=7500, gain_db=4.5),
            Compressor(threshold_db=-22, ratio=4.0, attack_ms=3.0, release_ms=100),
            Reverb(room_size=0.25, damping=0.5, wet_level=0.1),
            Gain(gain_db=6.0)
        ])
        vocal_processed = vocal_board(y_tuned, music_sample_rate)

        # 7. Alignment and Peak Normalization
        target_length = max(len(music_shaped), len(vocal_processed))
        music_aligned = np.zeros(target_length, dtype=np.float32)
        vocal_aligned = np.zeros(target_length, dtype=np.float32)

        music_aligned[:len(music_shaped)] = music_shaped
        vocal_aligned[:len(vocal_processed)] = vocal_processed

        music_norm = normalize_audio(music_aligned, target_peak=0.35)
        vocal_norm = normalize_audio(vocal_aligned, target_peak=0.90)

        # 8. Mix Summation
        mixed_raw = music_norm + vocal_norm

        # 9. Mixbus Mastering (Pedalboard)
        master_board = Pedalboard([
            Compressor(threshold_db=-12, ratio=2.5),
            Gain(gain_db=1.5)
        ])
        master_compressed = master_board(mixed_raw, music_sample_rate)

        # 10. Final Soft Saturation Limiting via np.tanh
        master_track = np.tanh(master_compressed)

        # Return Base64 Encoded Audio Stems
        return {
            "status": "success",
            "music_track": audio_to_base64(music_norm, music_sample_rate),
            "vocal_track": audio_to_base64(vocal_norm, music_sample_rate),
            "master_track": audio_to_base64(master_track, music_sample_rate)
        }
    except Exception as e:
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
# @title 7. Secure ngrok Tunnel Configuration
from google.colab import userdata
import getpass

# Fetch NGROK_AUTHTOKEN securely from Google Colab Secrets (Key icon in sidebar)
try:
    NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    NGROK_TOKEN = None

if not NGROK_TOKEN:
    print("NGROK_AUTHTOKEN not found in Colab Secrets.")
    print("Enter your ngrok auth token (from https://dashboard.ngrok.com/get-started/your-authtoken):")
    NGROK_TOKEN = getpass.getpass("NGROK_AUTHTOKEN (hidden): ")

if not NGROK_TOKEN:
    raise RuntimeError("NGROK_AUTHTOKEN is missing. Add it to Colab Secrets or enter it above.")

ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
public_url = ngrok.connect(8000)
print("\n" + "=" * 60)
print(f"🚀 ISAICRAFT GPU ENGINE LIVE AT: {public_url.public_url}")
print(f"👉 Set VITE_GPU_ENGINE_URL={public_url.public_url} in your frontend/.env")
print("=" * 60 + "\n")

In [ ]:
# @title 8. Launch Uvicorn Server
nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)